In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
conn = sqlite3.connect("../database/air_quality.db")

df = pd.read_sql_query(
    "SELECT * FROM air_quality",
    conn
)

conn.close()

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day

df = df.dropna(subset=["calculated_aqi"])

print(f"Loaded {len(df)} rows")
df.head()

## Random Forest — EPA AQI (0–500)

Predicts the U.S. EPA AQI (0–500) calculated via breakpoint interpolation
from pollutant concentrations (PM2.5, PM10, CO, NO₂, O₃, SO₂) and
time features (hour, day).

In [ ]:
X = df[["pm2_5", "pm10", "co", "no2", "o3", "so2", "hour", "day"]]
y = df["calculated_aqi"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} rows")
print(f"Test set:     {len(X_test)} rows")

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("Random Forest — EPA AQI (0–500)")
print(f"MAE:  {mean_absolute_error(y_test, pred_rf):.2f}")
print(f"RMSE: {mean_squared_error(y_test, pred_rf) ** 0.5:.2f}")
print(f"R2:   {r2_score(y_test, pred_rf):.4f}")

In [ ]:
joblib.dump(rf, "saved_models/random_forest_aqi_500.pkl")

loaded_model = joblib.load("saved_models/random_forest_aqi_500.pkl")
sample = X.iloc[[0]]
prediction = loaded_model.predict(sample)
print(f"feature_names_in_: {list(loaded_model.feature_names_in_)}")
print(f"Sample prediction: {prediction[0]:.1f} (actual: {y.iloc[0]})")

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(importance["Feature"], importance["Importance"], color="#1D4ED8")
plt.xlabel("Importance")
plt.title("Feature Importance — Random Forest (EPA AQI 0–500)")
plt.tight_layout()
plt.show()